In [2]:
import os
import pandas as pd
import numpy as np

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PMID_PROTEIN/ALL_PMID_PROTEIN.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 2. Load KG Sources

#### CKG

In [3]:
CKG = pd.read_csv(PROC_DIR + 'CKG/File_44_PMID_PROTEIN_CKG.csv')
CKG['Head'] = 'PMID:' + CKG['Head'].astype(str)
CKG

/tmp/ipykernel_2582854/2301201376.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  CKG = pd.read_csv(PROC_DIR + 'CKG/File_44_PMID_PROTEIN_CKG.csv')


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Tail_Alt_name,Head_ID_IS,Tail_ID_IS
0,PMID:20463971,MENTIONED_IN_PUBLICATION,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID
1,PMID:17555535,MENTIONED_IN_PUBLICATION,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID
2,PMID:15644143,MENTIONED_IN_PUBLICATION,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID
3,PMID:20214810,MENTIONED_IN_PUBLICATION,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID
4,PMID:20089835,MENTIONED_IN_PUBLICATION,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID
...,...,...,...,...,...,...,...,...,...
237292205,PMID:8536891,MENTIONED_IN_PUBLICATION,Q8N4M9,PMID,Protein,CKG,Mucin-5AC {ECO:0000305},PMID,Uniprot_ID
237292206,PMID:6429045,MENTIONED_IN_PUBLICATION,Q8N4M9,PMID,Protein,CKG,Mucin-5AC {ECO:0000305},PMID,Uniprot_ID
237292207,PMID:31591430,MENTIONED_IN_PUBLICATION,A0A096LPK9,PMID,Protein,CKG,Olfactory receptor 4N4C {ECO:0000305},PMID,Uniprot_ID
237292208,PMID:38259516,MENTIONED_IN_PUBLICATION,A0A096LPK9,PMID,Protein,CKG,Olfactory receptor 4N4C {ECO:0000305},PMID,Uniprot_ID


In [4]:
CKG.columns = CKG.columns.str.lower()
CKG.rename(columns={'tail_alt_name': 'tail_detail_name'}, inplace=True)
CKG['relation'] = 'PMID_Protein'
CKG['tail_type'] = 'Protein'
CKG['kg_type'] = 'Generalised'
CKG['kg_source'] = 'CKG'
CKG['species'] = 'Homo species'
CKG['head_detail_name'] = ''

CKG

,head,relation,tail,head_type,tail_type,kg_source,tail_detail_name,head_id_is,tail_id_is,kg_type,species,head_detail_name
0,PMID:20463971,PMID_Protein,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID,Generalised,Homo species,
1,PMID:17555535,PMID_Protein,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID,Generalised,Homo species,
2,PMID:15644143,PMID_Protein,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID,Generalised,Homo species,
3,PMID:20214810,PMID_Protein,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID,Generalised,Homo species,
4,PMID:20089835,PMID_Protein,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID,Generalised,Homo species,
...,...,...,...,...,...,...,...,...,...,...,...,...
237292205,PMID:8536891,PMID_Protein,Q8N4M9,PMID,Protein,CKG,Mucin-5AC {ECO:0000305},PMID,Uniprot_ID,Generalised,Homo species,
237292206,PMID:6429045,PMID_Protein,Q8N4M9,PMID,Protein,CKG,Mucin-5AC {ECO:0000305},PMID,Uniprot_ID,Generalised,Homo species,
237292207,PMID:31591430,PMID_Protein,A0A096LPK9,PMID,Protein,CKG,Olfactory receptor 4N4C {ECO:0000305},PMID,Uniprot_ID,Generalised,Homo species,
237292208,PMID:38259516,PMID_Protein,A0A096LPK9,PMID,Protein,CKG,Olfactory receptor 4N4C {ECO:0000305},PMID,Uniprot_ID,Generalised,Homo species,


In [5]:
SOURCE_DFS = [
              ('CKG', CKG)
             ]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    print(f"[{name}] {'duplicate columns: ' + str(dupes) if dupes else '✓ no duplicates'}")

[CKG] ✓ no duplicates


In [6]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df

Combined: 237,292,210 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,PMID:20463971,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
1,PMID:17555535,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
2,PMID:15644143,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
3,PMID:20214810,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
4,PMID:20089835,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
237292205,PMID:8536891,PMID_Protein,Q8N4M9,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,Mucin-5AC {ECO:0000305},Generalised,Homo species
237292206,PMID:6429045,PMID_Protein,Q8N4M9,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,Mucin-5AC {ECO:0000305},Generalised,Homo species
237292207,PMID:31591430,PMID_Protein,A0A096LPK9,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,Olfactory receptor 4N4C {ECO:0000305},Generalised,Homo species
237292208,PMID:38259516,PMID_Protein,A0A096LPK9,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,Olfactory receptor 4N4C {ECO:0000305},Generalised,Homo species


In [7]:
final_df = final_df[~final_df['head_detail_name'].isna()]
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,PMID:20463971,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
1,PMID:17555535,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
2,PMID:15644143,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
3,PMID:20214810,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
4,PMID:20089835,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
237292205,PMID:8536891,PMID_Protein,Q8N4M9,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,Mucin-5AC {ECO:0000305},Generalised,Homo species
237292206,PMID:6429045,PMID_Protein,Q8N4M9,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,Mucin-5AC {ECO:0000305},Generalised,Homo species
237292207,PMID:31591430,PMID_Protein,A0A096LPK9,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,Olfactory receptor 4N4C {ECO:0000305},Generalised,Homo species
237292208,PMID:38259516,PMID_Protein,A0A096LPK9,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,Olfactory receptor 4N4C {ECO:0000305},Generalised,Homo species


In [8]:
import pandas as pd
import numpy as np

# Vectorized hash using numpy — avoids Python string ops entirely
keys = (
    pd.util.hash_array(final_df['head'].to_numpy()) ^
    pd.util.hash_array(final_df['relation'].to_numpy()) * 2654435761 ^
    pd.util.hash_array(final_df['tail'].to_numpy()) * 40503
)

# Keep first occurrence of each hash
mask = ~pd.Series(keys).duplicated(keep='first').to_numpy()
final_df = final_df.iloc[mask].reset_index(drop=True)

print(f"After dedup: {len(final_df):,} rows")
final_df

After dedup: 237,058,221 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,PMID:20463971,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
1,PMID:17555535,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
2,PMID:15644143,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
3,PMID:20214810,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
4,PMID:20089835,PMID_Protein,P84085,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,ADP-ribosylation factor 5,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
237058216,PMID:8536891,PMID_Protein,Q8N4M9,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,Mucin-5AC {ECO:0000305},Generalised,Homo species
237058217,PMID:6429045,PMID_Protein,Q8N4M9,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,Mucin-5AC {ECO:0000305},Generalised,Homo species
237058218,PMID:31591430,PMID_Protein,A0A096LPK9,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,Olfactory receptor 4N4C {ECO:0000305},Generalised,Homo species
237058219,PMID:38259516,PMID_Protein,A0A096LPK9,PMID,<NA>,Protein,CKG,PMID,Uniprot_ID,,Olfactory receptor 4N4C {ECO:0000305},Generalised,Homo species


In [9]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df)

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7fef2f0c8b80>>
Traceback (most recent call last):
  File "/home/arushis/.local/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


KeyboardInterrupt: 

In [10]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df):,} rows → {OUT_PATH}")

Saved 237,058,221 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PMID_PROTEIN/ALL_PMID_PROTEIN.parquet


In [11]:
#